In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score

# Create dataset
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
y = y.reshape(-1,1)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42)

In [ ]:
import numpy as np

class MLP:
    def __init__(self, input_size, hidden_size):
        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, 1) * 0.1
        self.b2 = np.zeros((1,1))

    def relu(self, x):
        return np.maximum(0, x)

    def drelu(self, x):
        return (x > 0).astype(float)

    def sigmoid(self, x):
        return 1/(1+np.exp(-x))

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

1️⃣** L2 Regularization**

In [ ]:
def train_L2(model, X, y, lr=0.1, epochs=2000, lamb=0.01):
    m = len(X)
    for _ in range(epochs):
        output = model.forward(X)

        dZ2 = output - y
        dW2 = np.dot(model.a1.T, dZ2)/m + lamb*model.W2
        db2 = np.sum(dZ2)/m

        dA1 = np.dot(dZ2, model.W2.T)
        dZ1 = dA1 * model.drelu(model.z1)
        dW1 = np.dot(X.T, dZ1)/m + lamb*model.W1
        db1 = np.sum(dZ1)/m

        model.W2 -= lr*dW2
        model.b2 -= lr*db2
        model.W1 -= lr*dW1
        model.b1 -= lr*db1

**2️⃣ Dataset Augmentation**

In [ ]:
def augment_data(X, y):
    noise = np.random.normal(0,0.05,X.shape)
    X_aug = np.vstack([X, X+noise])
    y_aug = np.vstack([y,y])
    return X_aug, y_aug

**3️⃣ Adding Noise to Inputs**

In [ ]:
def train_with_input_noise(model, X, y, lr=0.1, epochs=2000):
    for _ in range(epochs):
        X_noisy = X + np.random.normal(0,0.05,X.shape)
        output = model.forward(X_noisy)
        dZ2 = output - y
        dW2 = np.dot(model.a1.T, dZ2)/len(X)
        db2 = np.sum(dZ2)/len(X)
        dA1 = np.dot(dZ2, model.W2.T)
        dZ1 = dA1 * model.drelu(model.z1)
        dW1 = np.dot(X_noisy.T, dZ1)/len(X)
        db1 = np.sum(dZ1)/len(X)

        model.W2 -= lr*dW2
        model.b2 -= lr*db2
        model.W1 -= lr*dW1
        model.b1 -= lr*db1

4️⃣ Dropout

In [ ]:
def train_dropout(model, X, y, lr=0.1, epochs=2000, keep_prob=0.8):
    for _ in range(epochs):
        output = model.forward(X)

        dropout_mask = (np.random.rand(*model.a1.shape) < keep_prob)/keep_prob
        model.a1 *= dropout_mask

        dZ2 = output - y
        dW2 = np.dot(model.a1.T, dZ2)/len(X)
        db2 = np.sum(dZ2)/len(X)

        dA1 = np.dot(dZ2, model.W2.T) * dropout_mask
        dZ1 = dA1 * model.drelu(model.z1)
        dW1 = np.dot(X.T, dZ1)/len(X)
        db1 = np.sum(dZ1)/len(X)

        model.W2 -= lr*dW2
        model.b2 -= lr*db2
        model.W1 -= lr*dW1
        model.b1 -= lr*db1

**5️⃣ Early Stopping**

In [ ]:
def train_early_stop(model, X_train, y_train, X_val, y_val, lr=0.1, epochs=5000):
    best_loss = float('inf')
    patience = 100
    wait = 0

    for epoch in range(epochs):
        output = model.forward(X_train)
        loss = np.mean((output - y_train)**2)

        val_output = model.forward(X_val)
        val_loss = np.mean((val_output - y_val)**2)

        if val_loss < best_loss:
            best_loss = val_loss
            wait = 0
        else:
            wait += 1
            if wait > patience:
                print("Early stopping at epoch", epoch)
                break

        dZ2 = output - y_train
        dW2 = np.dot(model.a1.T, dZ2)/len(X_train)
        db2 = np.sum(dZ2)/len(X_train)

        dA1 = np.dot(dZ2, model.W2.T)
        dZ1 = dA1 * model.drelu(model.z1)
        dW1 = np.dot(X_train.T, dZ1)/len(X_train)
        db1 = np.sum(dZ1)/len(X_train)

        model.W2 -= lr*dW2
        model.b2 -= lr*db2
        model.W1 -= lr*dW1
        model.b1 -= lr*db1

**6️⃣ Ensemble Method**

In [ ]:
def ensemble_predict(models, X):
    preds = [model.forward(X) for model in models]
    avg = np.mean(preds, axis=0)
    return (avg > 0.5).astype(int)

**7️⃣ Parameter Sharing (Simple Example)**

In [ ]:
shared_W1 = np.random.randn(2,10)*0.1

model1 = MLP(2,10)
model2 = MLP(2,10)

model1.W1 = shared_W1
model2.W1 = shared_W1